In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Data Interoperability with Unity Catalog

This project shows how to achieve seamless data interoperability between systems using Databricks Unity Catalog as a unified governance layer, using both Delta and Iceberg open table formats. The goal is to implement, manage and optimize a data environment where tables can be accessed by both Delta and Iceberg clients, enabling cross platform analytics and eliminating data silos.

## Multi-Tier Data Architecture with Format Interoperability

### Introduction
In this project, we'll build a complete multi-tier data architecture using both Iceberg and Delta formats with the following components:

Bronze tier: Raw data in Iceberg format
Silver tier: Cleaned/validated data in Delta format with UniForm enabled
Gold tier: Aggregated analytical data in Delta format with UniForm enabled
This pattern allows you to leverage the best of both worlds - native Iceberg support for external systems and Delta Lake's performance optimizations within Databricks, while enabling seamless access from Snowflake and other external client

## Bronze Layer: Raw Data in Iceberg Format
The bronze layer represents raw, unprocessed data. We'll use Iceberg format for this layer to demonstrate native Iceberg support.

%md
### Bronze Table Schema

Here's the schema for our bronze layer table that contains retail sales data:

| Column Name | Data Type | Description |
|-------------|-----------|-------------|
| transaction_id | STRING | Unique transaction identifier |
| store_id | INT | Store identifier |
| product_id | STRING | Product identifier |
| transaction_date | STRING | Date of transaction (format: YYYY-MM-DD) |
| transaction_time | STRING | Time of transaction (format: HH:MM:SS) |
| quantity | INT | Number of items purchased |
| unit_price | DECIMAL(10,2) | Price per unit |
| customer_id | STRING | Customer identifier |
| payment_method | STRING | Method of payment |
| transaction_status | STRING | Status of transaction (completed, refunded, etc.) |

Create the bronze table In Iceberg format, using the above schema. Include the appropriate clustering columns.

In [0]:
USE CATALOG databricksdemo;
USE SCHEMA bronze;

In [0]:
DROP TABLE IF EXISTS retail_sales_bronze;
CREATE TABLE IF NOT EXISTS retail_sales_bronze (
   transaction_id STRING,
   store_id INT,
   product_id STRING,
   transaction_date STRING,
   transaction_time STRING,
   quantity INT,
   unit_price DECIMAL(10,2),
   customer_id STRING,
   payment_method STRING,
   transaction_status STRING
) 
USING ICEBERG;

In [0]:
DESCRIBE EXTENDED retail_sales_bronze;

In [0]:
DESCRIBE DETAIL retail_sales_bronze;

In [0]:
ALTER TABLE retail_sales_bronze
SET TBLPROPERTIES (
    delta.enableDeletionVectors = 'false',
    delta.enableRowTracking = 'false'
)

In [0]:
ALTER TABLE retail_sales_bronze CLUSTER BY (store_id, transaction_date);

In [0]:
SHOW TBLPROPERTIES retail_sales_bronze;

In [0]:
DESCRIBE DETAIL retail_sales_bronze;

In [0]:
DESCRIBE EXTENDED retail_sales_bronze;

In [0]:
-- Insert sample data into the bronze table
INSERT INTO retail_sales_bronze VALUES
('TX-10001', 1, 'P-1234', '2024-07-01', '09:30:45', 3, 19.99, 'C-5678', 'credit_card', 'completed'),
('TX-10002', 1, 'P-2345', '2024-07-01', '10:15:22', 1, 49.99, 'C-6789', 'debit_card', 'completed'),
('TX-10003', 2, 'P-3456', '2024-07-01', '11:05:17', 2, 12.50, 'C-7890', 'cash', 'completed'),
('TX-10004', 2, 'P-1234', '2024-07-02', '13:45:08', 1, 19.99, 'C-8901', 'credit_card', 'refunded'),
('TX-10005', 3, 'P-4567', '2024-07-02', '14:22:36', 4, 9.99, 'C-9012', 'mobile_payment', 'completed'),
('TX-10006', 3, 'P-5678', '2024-07-02', '15:10:05', 2, 29.99, 'C-0123', 'credit_card', 'completed'),
('TX-10007', 1, 'P-6789', '2024-07-03', '09:05:12', 3, 14.99, 'C-1234', 'debit_card', 'completed'),
('TX-10008', 2, 'P-7890', '2024-07-03', '10:30:28', 1, 99.99, 'C-2345', 'credit_card', 'completed'),
('TX-10009', 3, 'P-8901', '2024-07-03', '12:15:33', 2, 24.99, 'C-3456', 'mobile_payment', 'completed'),
('TX-10010', 1, 'P-9012', '2024-07-04', '14:50:41', 5, 8.99, 'C-4567', 'cash', 'completed'),
('TX-10011', 2, 'P-1234', '2024-07-04', '16:25:19', 1, 19.99, 'C-5678', 'credit_card', 'completed'),
('TX-10012', 3, 'P-2345', '2024-07-04', '17:40:02', 2, 49.99, 'C-6789', 'debit_card', 'completed'),
('TX-10013', 1, 'P-3456', '2024-07-05', '09:15:55', 3, 12.50, 'C-7890', 'credit_card', 'completed'),
('TX-10014', 2, 'P-4567', '2024-07-05', '11:30:10', 2, 9.99, 'C-8901', 'mobile_payment', 'completed'),
('TX-10015', 3, 'P-5678', '2024-07-05', '13:20:48', 1, 29.99, 'C-9012', 'credit_card', 'refunded');

In [0]:
SELECT * FROM retail_sales_bronze;

%md
## Silver Layer: Validated/Cleaned Data in Delta Format

The silver layer contains validated and cleaned data. We'll use Delta format with Liquid Clustering and enable UniForm for interoperability.

### Data Quality and Transformation Requirements

For the silver layer, implement the following:
1. Convert transaction_date from STRING to DATE type
2. Combine transaction_date and transaction_time into a timestamp column
3. Filter out refunded transactions
4. Calculate total_amount (quantity * unit_price)
5. Enable Liquid Clustering for better query performance
6. Configure UniForm for Iceberg compatibility

Create the silver table using CTAS (CREATE TABLE AS SELECT).
- Implement the data quality checks and transformations listed above
- Use Delta format with Liquid Clustering
- Don't worry about UniForm configuration yet - we'll do that separately

In [0]:
DROP TABLE IF EXISTS retail_sales_silver

In [0]:
DROP TABLE IF EXISTS retail_sales_silver;

CREATE TABLE IF NOT EXISTS retail_sales_silver USING DELTA
  CLUSTER BY (store_id, transaction_date) AS
SELECT
  transaction_id,
  store_id,
  product_id,
  transaction_date::date,
  to_timestamp(concat_ws(' ', transaction_date, transaction_time)) timestamp,
  quantity,
  unit_price,
  quantity * unit_price AS total_amount,
  customer_id,
  payment_method,
  transaction_status
FROM
  retail_sales_bronze
WHERE
  transaction_status <> 'refunded';

In [0]:
SELECT *
FROM retail_sales_silver;

%md
### Enable UniForm for Silver Layer

Now let's configure the silver table for Iceberg compatibility using Universal Format (UniForm). Note that you will have to:
1. Disable deletion vectors (required for Iceberg compatibility)
1. Enable Universal Format for Iceberg


In [0]:
-- Disable deleteion vector
ALTER TABLE retail_sales_silver
SET TBLPROPERTIES (
    'delta.enableDeletionVectors' = 'false'
)

In [0]:
-- Enable Universal Format for Iceberg
ALTER TABLE retail_sales_silver
SET TBLPROPERTIES (
  'delta.universalFormat.enabledFormats' = 'iceberg',
  'delta.enableIcebergCompatV2' = 'true',
  'delta.columnMapping.mode' = 'name'
);

In [0]:
-- Verify configuration
SHOW TBLPROPERTIES retail_sales_silver;

## Gold Layer: Aggregated/Analytics Data

The gold layer contains aggregated and summarized data optimized for analytics queries. We'll also use Delta format with Liquid Clustering and UniForm for this layer.

Create a gold table with aggregated data:
- Implement aggregations to summarize sales by store, date, and product
- Use Delta format with Liquid Clustering
- Don't configure UniForm yet - we'll do that separately

In [0]:
CREATE OR REPLACE TABLE retail_sales_gold
  USING DELTA
  CLUSTER BY (store_id, transaction_date)
  AS
    SELECT
      store_id,
      transaction_date,
      product_id,
      COUNT(DISTINCT transaction_id) AS transaction_count,
      SUM(quantity) AS total_quantity_sold,
      SUM(total_amount) AS total_sales_amount,
      AVG(total_amount) AS avg_transaction_amount
    FROM retail_sales_silver
    GROUP BY store_id, transaction_date, product_id;

In [0]:
-- Verify the aggregated data
SELECT * FROM retail_sales_gold ORDER BY transaction_date, store_id, product_id;

Configure the gold table for Iceberg compatibility
1. Disable deletion vectors (required for Iceberg compatibility)
1. Enable Universal Format for Iceberg

In [0]:
-- Disable deleteion vector
ALTER TABLE retail_sales_gold
SET TBLPROPERTIES (
    'delta.enableDeletionVectors' = 'false'
)

In [0]:
-- Enable Universal Format for Iceberg
ALTER TABLE retail_sales_gold
SET TBLPROPERTIES (
    'delta.universalFormat.enabledFormats' = "iceberg",
    'delta.enableIcebergCompatV2' = 'true',
    'delta.columnMapping.mode' = "name"
)

In [0]:
DESCRIBE DETAIL retail_sales_gold;

In [0]:
SHOW TBLPROPERTIES retail_sales_gold;

## Conclusion

In this lab, we've implemented a complete multi-tier data architecture that:
- Uses native table formats optimized for different workloads
- Implements data quality checks and transformations
- Enables cross-platform compatibility with external systems like Snowflake
- Optimizes query performance using Liquid Clustering

This architecture forms the foundation for a modern data lakehouse that combines the best features of data lakes and data warehouses while maintaining interoperability with external systems.

## Next Steps: Accessing from Downstream Analytics Systems

With this setup complete, we've now created:
1. A bronze layer in native Iceberg format
2. Silver and gold layers in Delta format with UniForm enabled for Iceberg compatibility

This architecture allows:
- Databricks to efficiently process data using Delta's optimizations
- External engines like Snowflake to access the same data as if it were native Iceberg tables

## Powering Downstream Analytics using UC Managed Tables

Next, we'll show how to:
- Configure Snowflake to read from these tables using Iceberg format
- Write queries in Snowflake that access your Delta tables through UniForm
- Analyze performance implications of this cross-platform architecture

### Grant privileges to the service principal

Since our downstream process will connect to Databricks through the service principal, we must grant appropriate access to the appropriate data objects to the service principal.

Grant the `EXTERNAL USE SCHEMA`, `USE` and `SELECT` privileges to the service principal created previously (at the catalog or schema level).

In [0]:
-- Grant EXTERNAL USE SCHEMA privilege
GRANT EXTERNAL USE SCHEMA ON SCHEMA databricksdemo.bronze TO `e033f219-6616-4ecc-88ff-4674c93c419a`;

-- Grant USE CATALOG privilege
GRANT USE CATALOG ON CATALOG databricksdemo TO `e033f219-6616-4ecc-88ff-4674c93c419a`;

-- Grant USE SCHEMA privilege
GRANT USE SCHEMA ON SCHEMA databricksdemo.bronze TO `e033f219-6616-4ecc-88ff-4674c93c419a`;

-- Grant SELECT privileges
GRANT SELECT ON TABLE databricksdemo.bronze.retail_sales_bronze TO `e033f219-6616-4ecc-88ff-4674c93c419a`;
GRANT SELECT ON TABLE databricksdemo.bronze.retail_sales_silver TO `e033f219-6616-4ecc-88ff-4674c93c419a`;
GRANT SELECT ON TABLE databricksdemo.bronze.retail_sales_gold TO `e033f219-6616-4ecc-88ff-4674c93c419a`;

## Snowflake preparations

Connect to your Snowflake environment to perform the steps outlined in this section.

### 1. Create catalog integration in Snowflake

Run the following SQL command to create a catalog integration.

Note that this integration uses:
  - The Databricks workspace URL in the `CATALOG_URI` and `OAUTH_TOKEN_URI` (the authorization endpoint)
  - The Application ID from the previous step as the `OAUTH_CLIENT_ID`
  - The secret from the previous step as the `OAUTH_CLIENT_SECRET`
  - As before, substitute `<catalog>` and `<schema>` as per the setup cell output

```sql
CREATE OR REPLACE CATALOG INTEGRATION unity_catalog_int_oauth
CATALOG_SOURCE = ICEBERG_REST
TABLE_FORMAT = ICEBERG
CATALOG_NAMESPACE = '<schema>'
REST_CONFIG = (
    CATALOG_URI = 'https://{your-workspace-deployment-name}.cloud.databricks.com/api/2.1/unity-catalog/iceberg-rest'
    WAREHOUSE  = '<catalog>'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
)
REST_AUTHENTICATION = (
TYPE = OAUTH
    OAUTH_TOKEN_URI = 'https://{your-workspace-deployment-name}.cloud.databricks.com/oidc/v1/token'
    OAUTH_CLIENT_ID = 'application-id-for-your-service-principal'
    OAUTH_CLIENT_SECRET = 'secret-for-your-service-principal'
    OAUTH_ALLOWED_SCOPES = ('all-apis', 'sql')
)
ENABLED = TRUE
REFRESH_INTERVAL_SECONDS = 30;
```

### 2. Define Iceberg tables in Snowflake

Run the following SQL commands to define Iceberg tables:

```sql
CREATE OR REPLACE ICEBERG TABLE retail_sales_bronze
CATALOG = 'unity_catalog_int_oauth'
CATALOG_TABLE_NAME = 'retail_sales_bronze'
AUTO_REFRESH = TRUE;

CREATE OR REPLACE ICEBERG TABLE retail_sales_silver
CATALOG = 'unity_catalog_int_oauth'
CATALOG_TABLE_NAME = 'retail_sales_silver'
AUTO_REFRESH = TRUE;

CREATE OR REPLACE ICEBERG TABLE retail_sales_gold
CATALOG = 'unity_catalog_int_oauth'
CATALOG_TABLE_NAME = 'retail_sales_gold'
AUTO_REFRESH = TRUE;
```

%md
- These commands create Snowflake tables that map to the corresponding tables in Databricks Unity Catalog
- The `AUTO_REFRESH` parameter ensures that Snowflake automatically refreshes the table metadata at the interval specified in the catalog integration

In [0]:
%python
spark.conf.get("spark.databricks.workspaceUrl")

In [0]:
-- The URLs are incorrect. For Azure Databricks, use the workspace URL format:
-- https://adb-<workspace-id>.<region>.azuredatabricks.net

-- Correct CATALOG_URI:
-- 'https://adb-7405618162065172.12.azuredatabricks.net/api/2.1/unity-catalog/iceberg-rest'

-- Correct OAUTH_TOKEN_URI:
-- 'https://adb-7405618162065172.12.azuredatabricks.net/oidc/v1/token'

### 3. Query the Unity Catalog Managed Tables from Snowflake

You can now query the managed tables in Unity Catalog directly from Snowflake. The following cell provides some examples.

In [0]:
-- This is the managed Iceberg table
SELECT * FROM retail_sales_bronze;

-- These are the managed Delta tables
SELECT * FROM retail_sales_silver;
SELECT * FROM retail_sales_gold;

In [0]:
-- CREATE OR REPLACE CATALOG INTEGRATION unity_catalog_int_oauth
-- CATALOG_SOURCE = ICEBERG_REST
-- TABLE_FORMAT = ICEBERG
-- CATALOG_NAMESPACE = 'bronze'
-- REST_CONFIG = (
--     CATALOG_URI = 'https://adb-7405618162065172.12.azuredatabricks.net/api/2.1/unity-catalog/iceberg-rest'
--     WAREHOUSE  = 'databricksdemo'
--     ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
-- )
-- REST_AUTHENTICATION = (
-- TYPE = OAUTH
--     OAUTH_TOKEN_URI = 'https://adb-7405618162065172.12.azuredatabricks.net/oidc/v1/token'
--     OAUTH_CLIENT_ID = ''
--     OAUTH_CLIENT_SECRET = ''
--     OAUTH_ALLOWED_SCOPES = ('all-apis', 'sql')
-- )
-- ENABLED = TRUE
-- REFRESH_INTERVAL_SECONDS = 30;


-- CREATE OR REPLACE ICEBERG TABLE retail_sales_bronze
-- CATALOG = 'unity_catalog_int_oauth'
-- CATALOG_TABLE_NAME = 'retail_sales_bronze'
-- AUTO_REFRESH = TRUE;

CREATE OR REPLACE ICEBERG TABLE retail_sales_silver
CATALOG = 'unity_catalog_int_oauth'
CATALOG_TABLE_NAME = 'retail_sales_silver'
AUTO_REFRESH = TRUE;

-- CREATE OR REPLACE ICEBERG TABLE retail_sales_gold
-- CATALOG = 'unity_catalog_int_oauth'
-- CATALOG_TABLE_NAME = 'retail_sales_gold'
-- AUTO_REFRESH = TRUE;

-- This is the managed Iceberg table
-- SELECT * FROM retail_sales_bronze;

-- These are the managed Delta tables
SELECT * FROM retail_sales_silver;
-- SELECT * FROM retail_sales_gold;
